# Part A, Q1: Tokenisation and Filtering

This notebook tokenises the text in `Data_1.txt` using three different methods, compares them,
then removes stop words and punctuation from the chosen method's tokens.

The printed output backs the report's Q1 section directly:

| Report section | Produced by |
|---|---|
| 1.1 Word Tokenisation Methods | the three method outputs below |
| 1.2 Tokenisation Method Comparison | the token counts and attached-punctuation demonstration |
| 1.3 Stop Word and Punctuation Removal | the filtering section |

In [1]:
import re
import string
from pathlib import Path

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


# Make the notebook reproducible top-to-bottom: download the NLTK data it needs
# on first run instead of assuming it is already installed.
def ensure_nltk_data():
    resources = [
        ("tokenizers/punkt", "punkt"),
        ("tokenizers/punkt_tab", "punkt_tab"),   # required by NLTK >= 3.9
        ("corpora/stopwords", "stopwords"),
    ]
    for path, package in resources:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(package)


ensure_nltk_data()

In [2]:
# Read the same corpus used across Part A so the outputs are comparable.
# Walk up from the working directory so this runs whether the kernel starts in
# this folder, in Part_A, or at the repository root.
def find_data_file(filename):
    marker = Path("data") / filename
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate / marker
        if (candidate / "Part_A" / marker).exists():
            return candidate / "Part_A" / marker
    raise FileNotFoundError(f"Could not locate Part_A/data/{filename} from {Path.cwd()}")


DATA_FILE = find_data_file("Data_1.txt")
with open(DATA_FILE, "r", encoding="utf-8") as file:
    text = file.read()

print(f"Data file: {DATA_FILE}")
print(f"Characters read: {len(text)}")

Data file: C:\Users\yingx\Desktop\TextAssignment\Part_A\data\Data_1.txt
Characters read: 524


## 1.1 Word Tokenisation Methods

Three ways of cutting the same text into tokens, chosen because they behave differently at
exactly one point: what they do with punctuation.

1. **`str.split()`** splits on whitespace only, so punctuation stays glued to its neighbouring
   word (`"tasks,"`, `"classified."`).
2. **Regular expression `\b\w+\b`** matches runs of word characters only, so punctuation is
   discarded entirely and never becomes a token.
3. **NLTK `word_tokenize`** splits words but keeps punctuation as its own separate token, which
   is the most linguistically correct of the three.

In [3]:
# Method 1: Python's built-in str.split() -> splits on whitespace only.
# Punctuation stays glued to the neighbouring word (e.g. "tasks," "classified.").
split_tokens = text.split()

# Method 2: Regular Expression -> \b\w+\b matches runs of word characters only,
# so punctuation is stripped out entirely and never appears as a token.
regex_tokens = re.findall(r"\b\w+\b", text)

# Method 3: NLTK word_tokenize -> splits words but keeps punctuation as its own
# separate tokens (e.g. "tasks" and "," become two tokens), which is the most
# linguistically correct behaviour of the three.
nltk_tokens = word_tokenize(text)


print("=== Method 1: Python split() ===")
print(split_tokens)
print(f"Token count: {len(split_tokens)}")

print("\n=== Method 2: Regular Expression (\\b\\w+\\b) ===")
print(regex_tokens)
print(f"Token count: {len(regex_tokens)}")

print("\n=== Method 3: NLTK word_tokenize ===")
print(nltk_tokens)
print(f"Token count: {len(nltk_tokens)}")

=== Method 1: Python split() ===
['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input.', 'In', 'basic', 'classification', 'tasks,', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs,', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance.', 'The', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants.', 'For', 'example,', 'in', 'multiclass', 'classification,', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels;', 'in', 'open-class', 'classification,', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance;', 'and', 'in', 'sequence', 'classification,', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified.']
Token count: 82

=== Method 2: Regular Expression (\b\w+\b) ===
['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', 'In', '

## 1.2 Tokenisation Method Comparison

The token counts alone already tell the story: `split()` yields the fewest tokens because
punctuation is absorbed into words, while NLTK yields the most because punctuation is separated
out.

The cell below makes it concrete by listing the tokens where `split()` left punctuation
attached. These are the "dirty" tokens that would pollute a vocabulary: `"tasks,"` and
`"tasks"` would be counted as two unrelated words.

In [4]:
# Demonstrate the key difference in a concrete way: split() keeps punctuation
# attached to words, while regex and NLTK do not.
split_with_punct = [tok for tok in split_tokens if any(ch in string.punctuation for ch in tok)]

print("\n=== Comparison: tokens where split() left punctuation attached ===")
print(split_with_punct)
print(
    "\nOnly split() produces these 'dirty' tokens. Regex removes the punctuation, "
    "and NLTK isolates it as separate tokens, which is why NLTK word_tokenize is "
    "chosen as the most suitable method."
)


=== Comparison: tokens where split() left punctuation attached ===
['input.', 'tasks,', 'inputs,', 'advance.', 'variants.', 'example,', 'classification,', 'labels;', 'open-class', 'classification,', 'advance;', 'classification,', 'classified.']

Only split() produces these 'dirty' tokens. Regex removes the punctuation, and NLTK isolates it as separate tokens, which is why NLTK word_tokenize is chosen as the most suitable method.


## 1.3 Stop Word and Punctuation Removal

Filtering is applied to the NLTK tokens, since 1.2 selected that as the most suitable method.

Two things are removed:

- **English stop words**, compared in lowercase so that sentence-initial words such as `In`,
  `The` and `For` are caught as well as their lowercase forms.
- **Pure-punctuation tokens** such as `,` and `.`, which NLTK deliberately kept as separate
  tokens in the previous step.

Both the kept and the removed tokens are printed, so the report can point at exactly what the
filter did rather than just asserting it worked.

In [5]:
# Filter the NLTK tokens (the chosen method) by removing:
#   - English stop words (compared in lowercase so sentence-initial words like
#     "In", "The", "For" are also caught), and
#   - pure-punctuation tokens such as "," and ".".
stop_words = set(stopwords.words("english"))

filtered_tokens = [
    tok for tok in nltk_tokens
    if tok.lower() not in stop_words and tok not in string.punctuation
]

# Show exactly which tokens were dropped, so the report can point at them.
removed_tokens = [
    tok for tok in nltk_tokens
    if tok.lower() in stop_words or tok in string.punctuation
]

print("\n=== 1.3 After Stop Word and Punctuation Removal ===")
print("Tokens removed (stop words + punctuation):")
print(removed_tokens)
print(f"\nRemoved count: {len(removed_tokens)}")

print("\nFiltered tokens (kept):")
print(filtered_tokens)
print(f"Filtered token count: {len(filtered_tokens)}")
print(
    f"\nSummary: {len(nltk_tokens)} original NLTK tokens -> "
    f"{len(filtered_tokens)} tokens after filtering "
    f"({len(removed_tokens)} removed)."
)


=== 1.3 After Stop Word and Punctuation Removal ===
Tokens removed (stop words + punctuation):
['is', 'the', 'of', 'the', 'for', 'a', '.', 'In', ',', 'each', 'is', 'in', 'from', 'all', 'other', ',', 'and', 'the', 'of', 'is', 'in', '.', 'The', 'has', 'a', 'of', '.', 'For', ',', 'in', ',', 'each', 'be', ';', 'in', ',', 'the', 'of', 'is', 'not', 'in', ';', 'and', 'in', ',', 'a', 'of', 'are', '.']

Removed count: 49

Filtered tokens (kept):
['Classification', 'task', 'choosing', 'correct', 'class', 'label', 'given', 'input', 'basic', 'classification', 'tasks', 'input', 'considered', 'isolation', 'inputs', 'set', 'labels', 'defined', 'advance', 'basic', 'classification', 'task', 'number', 'interesting', 'variants', 'example', 'multiclass', 'classification', 'instance', 'may', 'assigned', 'multiple', 'labels', 'open-class', 'classification', 'set', 'labels', 'defined', 'advance', 'sequence', 'classification', 'list', 'inputs', 'jointly', 'classified']
Filtered token count: 45

Summary: 94 o